# note: repeat this pipeline on whole brain roi; see direction along corpus collosum (QC)

In [ ]:
"""
# image given is 4d
img = nib.load(t1_anat)
data = img.get_fdata()
vol3d = data.mean(axis = 3)
img3d = nib.Nifti1Image(vol3d, img.affine)
nib.save(img3d, os.path.join(path, 'processing/T1-2.5mm_3d.nii'))
fixed = os.path.join(path, 'processing/T1-2.5mm_3d.nii')
"""

In [ ]:
# 1. suit isolate
import SUITPy as suit
import os
import nibabel as nib
import numpy as np

t1_anat = os.path.join('')
save_folder = os.path.join('')

# image given is 4d
img = nib.load(t1_anat)

suit.isolate(t1_file = t1_anat, result_folder = save_folder, verbose = 0)

In [ ]:
# 2. make transformation file (save inverse)
mask_path = ''
template_space = 'MNI152NLin2009cSymC'

results = suit.normalize(
    source_file = t1_anat,
    mask_file = str(mask_path),
    space = template_space,
    write_inv_deformation = True,

    result_folder = save_folder
    )

In [ ]:
# 3. back transform atlas to native space
nettekoven_img = suit.reslice_image(
    'atl-NettekovenSym32_space-MNISym_dseg.nii',
    deformation = results['inv_deformation'],
    interp = 0,
    voxelsize = (1.1, 1.1, 1.1)
    )
nib.save(nettekoven_img, os.path.join(save_folder, ''))

In [ ]:
# 4. principal eigenvector direction in cerebellar cortex

import numpy as np
import os
import nibabel as nib
from nibabel.processing import resample_from_to
from dipy.io.gradients import read_bvals_bvecs # read b values, vectors (b = degree of diffusion weighing)
from dipy.core.gradients import gradient_table # read gradient table
from dipy.reconst.dti import TensorModel # solve Stejskal-Tanner equation for diffusion tensor, D

dwi_dir = os.path.join('')
img = nib.load() # image showing water diffusion along gradients
data = img.get_fdata()

bvals, bvecs = read_bvals_bvecs('bval', 'bvec') # b value, b vector


# roi
roi = nib.load(os.path.join('atl-NettekovenSym32_space-native_cerebellum_dseg.nii'))
roi_bin = nib.Nifti1Image((roi.get_fdata() > 0).astype(np.float32), roi.affine)
roi_dwi = resample_from_to(roi_bin, (img.shape[:3], img.affine), order=1)
roi_mask = roi_dwi.get_fdata() > 0.5



# fit in only the roi mask
grad_val = 2000 # matches bval, bvec gradient
keep = (bvals < 50) | (np.abs(bvals - grad_val) < 50) 
gtab = gradient_table(bvals[keep], bvecs = bvecs[keep], b0_threshold = 50) # pair b-values with their vectors

fit = TensorModel(gtab, fit_method = 'WLS').fit(data[..., keep], mask = roi_mask) # solve Stejskal-Tanner equation for diffusion tensor D using weighted least squres

fa, v1 = fit.fa, fit.evecs[..., :, 0] # fa, first eigenvector

# should I scale by FA? Noisier with lower FA (e.g. crossing fibres)

In [ ]:
# save v1 image
v1_img = nib.Nifti1Image(v1, img.affine)

v1_save_name = '...cerebellar_cortex_v1.nii'

nib.save(v1_img, f'{save_folder}/{v1_save_name}') # EDIT NAME

In [ ]:
from dipy.reconst.dti import color_fa

# colour map for ROI
rgb = color_fa(fa, fit.evecs)
rgb_roi = rgb * roi_mask[..., None] # only inside ROI

rgb8 = (np.clip(rgb_roi, 0, 1) * 255).astype(np.uint8)
rgb_dtype = np.dtype([('R', 'u1'), ('G', 'u1'), ('B', 'u1')])
rgb_struct = np.ascontiguousarray(rgb8).view(rgb_dtype).reshape(rgb8.shape[:3])

rgb_save_name = '...cerebellar_cortex_v1_rgb.nii'

nib.save(nib.Nifti1Image(rgb_struct.astype(np.float32), img.affine), f'{save_folder}/{rgb_save_name}')